In [48]:
# handwrite_digit_bayes_improved.py
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.ensemble import VotingClassifier
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.feature_selection import VarianceThreshold
import joblib
import time


class DigitRecognizer:
    def __init__(self, n_components=200, random_state=42):
        self.n_components = n_components
        self.random_state = random_state
        self.model = None
        self.pca = None
        self.scaler = None
        self.feature_selector = None
        self.accuracy = 0.0

    def load_data(self, data_path):
        df = pd.read_csv(data_path)


        X = df.iloc[:, 1:].values
        y = df.iloc[:, 0].values

        print(f"样本数量: {len(X)}")
        print(f"特征维度: {X.shape[1]}")
        print(f"类别分布: {dict(zip(*np.unique(y, return_counts=True)))}")

        return X, y

    def advanced_feature_engineering(self, X, y, test_size=0.2):
        X_normalized = X / 255.0

        self.feature_selector = VarianceThreshold(threshold=0.001)
        X_selected = self.feature_selector.fit_transform(X_normalized)
        print(f"方差特征选择: 784 -> {X_selected.shape[1]} 个特征")

        self.pca = PCA(n_components=self.n_components, random_state=self.random_state)
        X_pca = self.pca.fit_transform(X_selected)

        self.scaler = StandardScaler()
        X_scaled = self.scaler.fit_transform(X_pca)

        print(f"PCA降维: {X_selected.shape[1]} -> {X_pca.shape[1]} 个特征")
        print(f"PCA保留方差比例: {self.pca.explained_variance_ratio_.sum():.4f}")

        X_train, X_test, y_train, y_test = train_test_split(
            X_scaled, y, test_size=test_size, random_state=self.random_state, stratify=y
        )

        print(f"- 训练集: {X_train.shape}")
        print(f"- 测试集: {X_test.shape}")

        return X_train, X_test, y_train, y_test

    def create_ensemble_model(self):
        estimators = [
            ('gaussian_nb', GaussianNB(var_smoothing=1e-9)),
            ('bernoulli_nb', BernoulliNB(alpha=0.1, binarize=0.3)),
            ('gaussian_nb2', GaussianNB(var_smoothing=1e-7)),
        ]

        ensemble_model = VotingClassifier(
            estimators=estimators,
            voting='soft',
            n_jobs=-1
        )

        return ensemble_model

    def train_and_evaluate(self, X_train, X_test, y_train, y_test):
        start_time = time.time()

        self.model = self.create_ensemble_model()
        self.model.fit(X_train, y_train)

        training_time = time.time() - start_time
        print(f"模型训练完成! 耗时: {training_time:.2f}秒")

        train_accuracy = self.model.score(X_train, y_train)
        print(f"训练集准确率: {train_accuracy:.4f}")

        y_pred = self.model.predict(X_test)
        self.accuracy = accuracy_score(y_test, y_pred)

        print(f"测试集准确率: {self.accuracy:.4f}")
        print(f"错误分类数: {np.sum(y_pred != y_test)}/{len(y_test)}")

        self.detailed_evaluation(y_test, y_pred)

        return y_pred

    def detailed_evaluation(self, y_true, y_pred):
        print(classification_report(y_true, y_pred))

        cm = confusion_matrix(y_true, y_pred)
        print("混淆矩阵 (行:真实标签, 列:预测标签):")
        print("   " + " ".join(f"{i:2d}" for i in range(10)))
        print("  " + "-" * 30)
        for i in range(10):
            print(f"{i} | " + " ".join(f"{cm[i, j]:2d}" for j in range(10)))

        print("\n最易混淆的数字对 (错误数 > 10):")
        confusion_pairs = []
        for i in range(10):
            for j in range(10):
                if i != j and cm[i, j] > 10:
                    confusion_pairs.append((i, j, cm[i, j]))

        confusion_pairs.sort(key=lambda x: x[2], reverse=True)
        for i, j, count in confusion_pairs[:10]:
            print(f"  {i} → {j}: {count} 次错误")

    def cross_validation(self, X, y, cv=5):
        print(f"{cv} 折交叉验证")

        X_normalized = X / 255.0
        if self.feature_selector is not None:
            X_selected = self.feature_selector.transform(X_normalized)
        else:
            X_selected = X_normalized

        X_pca = self.pca.transform(X_selected)
        X_scaled = self.scaler.transform(X_pca)

        model = self.create_ensemble_model()
        scores = cross_val_score(model, X_scaled, y, cv=cv, scoring='accuracy', n_jobs=-1)

        print(f"各折准确率: {[f'{s:.4f}' for s in scores]}")
        print(f"平均准确率: {scores.mean():.4f} (±{scores.std():.4f})")

        return scores

    def predict_single(self, pixel_list):
        if self.model is None:
            raise ValueError("请先训练模型!")

        pixel_array = np.array(pixel_list, dtype=np.float32).reshape(1, -1)
        pixel_normalized = pixel_array / 255.0

        if self.feature_selector is not None:
            pixel_selected = self.feature_selector.transform(pixel_normalized)
        else:
            pixel_selected = pixel_normalized

        pixel_pca = self.pca.transform(pixel_selected)
        pixel_scaled = self.scaler.transform(pixel_pca)

        prediction = self.model.predict(pixel_scaled)[0]
        probability = self.model.predict_proba(pixel_scaled)[0]

        return prediction, probability

    def predict_batch(self, pixel_lists):
        if self.model is None:
            raise ValueError("请先训练模型!")

        pixel_array = np.array(pixel_lists, dtype=np.float32)
        pixel_normalized = pixel_array / 255.0

        if self.feature_selector is not None:
            pixel_selected = self.feature_selector.transform(pixel_normalized)
        else:
            pixel_selected = pixel_normalized

        pixel_pca = self.pca.transform(pixel_selected)
        pixel_scaled = self.scaler.transform(pixel_pca)

        predictions = self.model.predict(pixel_scaled)
        probabilities = self.model.predict_proba(pixel_scaled)

        return predictions, probabilities

model = DigitRecognizer(n_components=100, random_state=42)

try:
    data_path = './digit-recognizer/train.csv'
    X, y = model.load_data(data_path)

    X_train, X_test, y_train, y_test = model.advanced_feature_engineering(X, y, test_size=0.2)

    y_pred = model.train_and_evaluate(X_train, X_test, y_train, y_test)

    model.cross_validation(X, y, cv=5)

    print(f"最终准确率: {model.accuracy:.4f}")

except FileNotFoundError:
    print(f"错误：未找到数据文件!")
except Exception as e:
    print(f"未知错误: {str(e)}")
    import traceback

    traceback.print_exc()

改进版手写数字识别 - 贝叶斯分类器
数据加载完成!
样本数量: 42000
特征维度: 784
类别分布: {np.int64(0): np.int64(4132), np.int64(1): np.int64(4684), np.int64(2): np.int64(4177), np.int64(3): np.int64(4351), np.int64(4): np.int64(4072), np.int64(5): np.int64(3795), np.int64(6): np.int64(4137), np.int64(7): np.int64(4401), np.int64(8): np.int64(4063), np.int64(9): np.int64(4188)}
方差特征选择: 784 -> 545 个特征
PCA降维: 545 -> 100 个特征
PCA保留方差比例: 0.9156
数据划分完成:
- 训练集: (33600, 100)
- 测试集: (8400, 100)

创建集成贝叶斯模型...
模型训练完成! 耗时: 0.27秒

模型评估结果:
训练集准确率: 0.8727
测试集准确率: 0.8715
错误分类数: 1079/8400

详细分类报告:
              precision    recall  f1-score   support

           0       0.96      0.94      0.95       827
           1       0.98      0.94      0.96       937
           2       0.77      0.87      0.81       835
           3       0.85      0.83      0.84       870
           4       0.88      0.84      0.86       814
           5       0.78      0.84      0.81       759
           6       0.93      0.89      0.91       827
           7  

In [39]:
test_df = pd.read_csv('./digit-recognizer/test.csv')
X_test = test_df.iloc[:, :784].values

predictions, possible = model.predict_batch(X_test)

print(predictions, possible)

test_df.insert(0, 'Label', predictions)
test_df.to_csv('test_predictions.csv', index=False)



KeyboardInterrupt: 

In [88]:
from load_proto import simple_digit_processing_quick

# str = '0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	80	218	254	254	163	14	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	6	188	254	253	253	253	253	122	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	11	213	253	254	236	111	238	253	238	31	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	15	178	253	235	26	19	0	227	253	245	41	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	118	253	230	23	0	0	47	241	253	218	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	30	244	250	105	0	0	59	229	253	253	133	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	105	253	235	1	27	115	248	253	253	248	39	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	87	253	249	194	253	254	253	253	253	107	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	22	196	244	211	156	223	253	253	208	10	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	254	253	253	26	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	116	255	254	108	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	33	250	254	161	7	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	5	164	253	235	6	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	36	253	253	48	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	14	212	253	192	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	5	148	253	245	14	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	75	253	251	128	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	19	207	253	199	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	111	253	243	62	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	19	230	253	166	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0'

# pixels = str.split('	')
# 
# X_test = np.array(pixels)

# X_test = simple_digit_processing_quick('/Users/Liuhaixin/Downloads/9-2.png')

for i in range(1,10):
    X_test = simple_digit_processing_quick(f'{i}.png')
    
    predictions, possible = model.predict_single(X_test)
    
    print(predictions, possible)

1 [4.92154506e-05 9.85490581e-01 3.19474550e-05 2.35791692e-05
 8.63113532e-05 3.66774976e-04 4.19882193e-03 8.37374454e-05
 7.67810221e-05 9.59225065e-03]
2 [2.12715410e-04 3.93163655e-04 8.48678306e-01 5.53635034e-02
 8.79770929e-06 8.80601679e-04 9.10785607e-02 7.04176482e-04
 2.42129029e-03 2.58884252e-04]
3 [6.55902551e-03 1.85888820e-03 9.48729113e-02 6.45006330e-01
 1.80418552e-05 1.58948484e-01 3.62945564e-02 1.97634488e-03
 2.31330546e-02 3.13323639e-02]
4 [7.30435889e-04 2.43267342e-03 1.52863381e-03 4.84709247e-03
 7.68232573e-01 7.78378062e-03 1.09042859e-02 1.01419435e-02
 3.12970638e-03 1.90268875e-01]
5 [2.43958802e-03 1.41211443e-08 1.93935491e-03 1.71541583e-01
 2.54247204e-02 5.16407408e-01 1.95149084e-03 5.16364308e-02
 2.11983495e-01 1.66759153e-02]
6 [1.88114600e-03 5.38875501e-05 1.71490470e-03 1.54747869e-04
 5.45011937e-03 7.72113242e-02 8.84809556e-01 1.26141023e-03
 1.24521811e-02 1.50107227e-02]
2 [6.61969968e-03 2.02591756e-01 6.68923498e-01 2.26576185e-02
 